# Лабораторная работа 6
# Ансамбли моделей машинного обучения.
# Задание
1. Выберите набор данных (датасет) для решения задачи классификации или регресии.
2. В случае необходимости проведите удаление или заполнение пропусков и кодирование категориальных признаков.
3. С использованием метода train_test_split разделите выборку на обучающую и тестовую.
4. Обучите следующие ансамблевые модели:
 - одну из моделей группы стекинга.
 - модель многослойного персептрона. По желанию, вместо библиотеки scikit-learn возможно использование библиотек TensorFlow, PyTorch или других аналогичных библиотек.
 - двумя методами на выбор из семейства МГУА (один из линейных методов COMBI / MULTI + один из нелинейных методов MIA / RIA) с использованием библиотеки gmdh.
 - В настоящее время библиотека МГУА не позволяет решать задачу классификации !!!


In [4]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
sns.set(style="ticks")
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC

# Импорт данных

In [5]:
data = pd.read_csv('diabetes_binary_5050split_health_indicators_BRFSS2015.csv')

In [6]:
data.head()

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,3.0,5.0,30.0,0.0,1.0,4.0,6.0,8.0
1,0.0,1.0,1.0,1.0,26.0,1.0,1.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,12.0,6.0,8.0
2,0.0,0.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,10.0,0.0,1.0,13.0,6.0,8.0
3,0.0,1.0,1.0,1.0,28.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,3.0,0.0,3.0,0.0,1.0,11.0,6.0,8.0
4,0.0,0.0,0.0,1.0,29.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,8.0,5.0,8.0


# Обработка данных

Типы данных в представленном датасете:

In [7]:
data.dtypes

Diabetes_binary         float64
HighBP                  float64
HighChol                float64
CholCheck               float64
BMI                     float64
Smoker                  float64
Stroke                  float64
HeartDiseaseorAttack    float64
PhysActivity            float64
Fruits                  float64
Veggies                 float64
HvyAlcoholConsump       float64
AnyHealthcare           float64
NoDocbcCost             float64
GenHlth                 float64
MentHlth                float64
PhysHlth                float64
DiffWalk                float64
Sex                     float64
Age                     float64
Education               float64
Income                  float64
dtype: object

Проверим, есть ли в датасете пропущенные значения:

In [8]:
data.isnull().sum()

Diabetes_binary         0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
dtype: int64

В данном датасете нет строк или столбцов, содержащих пропущенные значения.

In [9]:
data.columns.tolist()

['Diabetes_binary',
 'HighBP',
 'HighChol',
 'CholCheck',
 'BMI',
 'Smoker',
 'Stroke',
 'HeartDiseaseorAttack',
 'PhysActivity',
 'Fruits',
 'Veggies',
 'HvyAlcoholConsump',
 'AnyHealthcare',
 'NoDocbcCost',
 'GenHlth',
 'MentHlth',
 'PhysHlth',
 'DiffWalk',
 'Sex',
 'Age',
 'Education',
 'Income']

In [10]:
data.head()

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,3.0,5.0,30.0,0.0,1.0,4.0,6.0,8.0
1,0.0,1.0,1.0,1.0,26.0,1.0,1.0,0.0,0.0,1.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,12.0,6.0,8.0
2,0.0,0.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,1.0,0.0,10.0,0.0,1.0,13.0,6.0,8.0
3,0.0,1.0,1.0,1.0,28.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,3.0,0.0,3.0,0.0,1.0,11.0,6.0,8.0
4,0.0,0.0,0.0,1.0,29.0,1.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,8.0,5.0,8.0


Проверим, содержатся ли в данных дубликаты:

In [11]:
print(data.duplicated())

0        False
1        False
2        False
3        False
4        False
         ...  
70687    False
70688    False
70689    False
70690    False
70691    False
Length: 70692, dtype: bool


duplicate_rows = data[data.duplicated()]
print(duplicate_rows)

В данном датасете содержатся дубликаты. Удалим эти строки:

In [12]:
data = data.drop_duplicates()

In [13]:
print(data.duplicated())
duplicate_rows = data[data.duplicated()]
print(duplicate_rows)

0        False
1        False
2        False
3        False
4        False
         ...  
70687    False
70688    False
70689    False
70690    False
70691    False
Length: 69057, dtype: bool
Empty DataFrame
Columns: [Diabetes_binary, HighBP, HighChol, CholCheck, BMI, Smoker, Stroke, HeartDiseaseorAttack, PhysActivity, Fruits, Veggies, HvyAlcoholConsump, AnyHealthcare, NoDocbcCost, GenHlth, MentHlth, PhysHlth, DiffWalk, Sex, Age, Education, Income]
Index: []

[0 rows x 22 columns]


Теперь дубликатов нет.

In [14]:
data['Income'].unique()

array([8., 7., 6., 3., 4., 1., 5., 2.])

In [15]:
le = LabelEncoder()
data['Income'] = le.fit_transform(data['Income'])

In [16]:
data['Income'].unique()

array([7, 6, 5, 2, 3, 0, 4, 1])

# Разделение выборки на обучающую и тестовую

С использованием метода train_test_split разделим выборку на обучающую и тестовую:

In [17]:
data.shape

(69057, 22)

In [18]:
# Признаки без целевой переменной
x = data.drop('Diabetes_binary', axis=1)
x.head()

,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,1.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,0.0,1.0,...,1.0,0.0,3.0,5.0,30.0,0.0,1.0,4.0,6.0,7
1,1.0,1.0,1.0,26.0,1.0,1.0,0.0,0.0,1.0,0.0,...,1.0,0.0,3.0,0.0,0.0,0.0,1.0,12.0,6.0,7
2,0.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,1.0,1.0,...,1.0,0.0,1.0,0.0,10.0,0.0,1.0,13.0,6.0,7
3,1.0,1.0,1.0,28.0,1.0,0.0,0.0,1.0,1.0,1.0,...,1.0,0.0,3.0,0.0,3.0,0.0,1.0,11.0,6.0,7
4,0.0,0.0,1.0,29.0,1.0,0.0,0.0,1.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,8.0,5.0,7


In [19]:
# Целевая переменная
y = data['Diabetes_binary']
y.head()

0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: Diabetes_binary, dtype: float64

In [20]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=1)
#20% данных будут отложены для тестирования, а 80% будут использованы для обучения модели

In [21]:
# Размер обучающей выборки
x_train.shape, y_train.shape

((55245, 21), (55245,))

In [22]:
# Размер тестовой выборки
x_test.shape, y_test.shape

((13812, 21), (13812,))

# Обучение ансамблевых моделей

## Стекинг

In [23]:
estimators = [
     ('rf', RandomForestClassifier(n_estimators=10, random_state=10)),
     ('svc', make_pipeline(StandardScaler(),
                           SVC(random_state=10)))]
stk_model = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression())
stk_model.fit(x_train, y_train)

StackingClassifier(estimators=[('rf',
                                RandomForestClassifier(n_estimators=10,
                                                       random_state=10)),
                               ('svc',
                                Pipeline(steps=[('standardscaler',
                                                 StandardScaler()),
                                                ('svc',
                                                 SVC(random_state=10))]))],
                   final_estimator=LogisticRegression())

## Многослойный персептрон

In [24]:
from sklearn.neural_network import MLPClassifier

In [25]:
model = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=100)
model.fit(x_train, y_train)

/Users/jerry/BMSTU/ТМО/venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=100)

## COMBI

In [ ]:
from gmdh import Combi, Mia
x_trainL=x_train.to_numpy()
y_trainL=y_train.to_numpy()

In [ ]:
comb = Combi()
comb.fit(x_trainL, y_trainL,verbose=1, limit=10)

LEVEL 1  [=========================] 100% :00s] (21 combinations) error=5827.577763                                 
LEVEL 2  [=========================] 100% :00s] (210 combinations) error=5342.559896                                
LEVEL 3  [=========================] 100% :01s] (1330 combinations) error=5194.894263                               
LEVEL 4  [=========================] 100% :14s] (5985 combinations) error=5003.578633                               
LEVEL 5  [=========================] 100% :02s] (20349 combinations) error=4923.763646                              
LEVEL 6  [=========================] 100% :15s] (54264 combinations) error=4906.477924                              
LEVEL 7  [=========================] 100% :59s] (116280 combinations) error=4890.701884                             
LEVEL 8  [=========================] 100% :43s] (203490 combinations) error=4877.698238                             
LEVEL 9  [=========================] 100% :38s] (293930 combinat

## MIA

In [ ]:
mia = Mia()
mia.fit(x_trainL, y_trainL, verbose=1, limit=0.001)

LEVEL 1  [=========================] 100% :00s] (210 combinations) error=5318.959863                                
LEVEL 2  [=========================] 100% :00s] (3 combinations) error=5213.268146                                  
LEVEL 3  [=========================] 100% :00s] (3 combinations) error=5147.310104                                  
LEVEL 4  [=========================] 100% :00s] (3 combinations) error=5135.161246                                  
LEVEL 5  [=========================] 100% :00s] (3 combinations) error=5129.677711                                  
LEVEL 6  [=========================] 100% :00s] (3 combinations) error=5128.577602                                  
LEVEL 7  [=========================] 100% :00s] (3 combinations) error=5126.391278                                  
LEVEL 8  [=========================] 100% :00s] (3 combinations) error=5126.697872                                  


# Оценка качества моделей

In [ ]:
models = [stk_model, model]

for model in models:
    print(f"{type(model).__name__}:")
    print('\t',f"accuracy = {accuracy_score(y_test,model.predict(x_test))}")


StackingClassifier:
	 accuracy = 0.7440631335070953
MLPClassifier:
	 accuracy = 0.742035910802201


In [ ]:
modelsL = [comb, mia]
x_testL=x_test.to_numpy()
y_testL=y_test.to_numpy()
linear_gmdh_pred = comb.predict(x_testL)
nonlinear_gmdh_pred = mia.predict(x_testL)
linear_gmdh_mse = mean_squared_error(y_testL, linear_gmdh_pred)
nonlinear_gmdh_mse = mean_squared_error(y_testL, nonlinear_gmdh_pred)
linear_gmdh_r2 = r2_score(y_testL, linear_gmdh_pred)
nonlinear_gmdh_r2 = r2_score(y_testL, nonlinear_gmdh_pred)
print(f"Linear GMDH Regressor MSE: {linear_gmdh_mse}, R2: {linear_gmdh_r2}")
print(f"Nonlinear GMDH Regressor MSE: {nonlinear_gmdh_mse}, R2: {nonlinear_gmdh_r2}")

Linear GMDH Regressor MSE: 0.17728465211278463, R2: 0.2908326042467564
Nonlinear GMDH Regressor MSE: 0.1857799793187071, R2: 0.25684991596044593
